In [ ]:
# This cell is mandatory in all Dymos documentation notebooks.
missing_packages = []
try:
    import openmdao.api as om  # noqa: F401
except ImportError:
    if 'google.colab' in str(get_ipython()):
        !python -m pip install openmdao[notebooks]
    else:
        missing_packages.append('openmdao')
try:
    import dymos as dm  # noqa: F401
except ImportError:
    if 'google.colab' in str(get_ipython()):
        !python -m pip install dymos
    else:
        missing_packages.append('dymos')
try:
    import pyoptsparse  # noqa: F401
except ImportError:
    if 'google.colab' in str(get_ipython()):
        !pip install -q condacolab
        import condacolab
        condacolab.install_miniconda()
        !conda install -c conda-forge pyoptsparse
    else:
        missing_packages.append('pyoptsparse')
if missing_packages:
    raise EnvironmentError('This notebook requires the following packages '
                           'please install them and restart this notebook\'s runtime: {",".join(missing_packages)}')

# Costate Estimation

For an optimal control problem with the dynamics $\dot{x} = f(x, u, t)$, the costates $\lambda(t)$ are the multipliers of the dynamics.
With an integrated cost carried as one of the states, the Hamiltonian is $H = \lambda^T f$, and $\lambda(t)$ is the sensitivity of the optimal objective to a change of the state at time $t$.
The costate of an integrated cost state is therefore one.

Dymos transcribes the dynamics into defect constraints at the nodes of each phase.
The function `estimate_costates` recovers estimates of the costates from the Lagrange multipliers of these defect constraints {cite}`costates-fahroo2001costate`.
The multipliers are computed after the optimization by OpenMDAO's `Driver.compute_lagrange_multipliers`, so the optimizer does not need to return them.

Costate estimates can be used to check the necessary conditions for optimality, for example that the Hamiltonian is constant along the solution of a problem that does not depend on time explicitly.
They are also needed by scaling methods that balance the states against the costates.

## Mapping the multipliers to costates

The optimizer's Lagrangian is $J + \mu^T c$, where $c$ are the constraints and $\mu$ their multipliers.

**Radau and Gauss-Lobatto.** At collocation node $k$ the defect is

$$
\Delta_k = \left(\frac{dx}{d\tau}\right)_k - f_k \left(\frac{dt}{d\tau}\right)_k
$$

where $\tau$ is the segment time on $[-1, 1]$.
With the quadrature weight $w_k$ of node $k$ in its segment, the costate estimate is

$$
\lambda_k = -\frac{\mu_k}{w_k}
$$

In the Radau transcription the last node of a phase is not a collocation node.
The costate there follows from stationarity of the Lagrangian with respect to the final state {cite}`costates-garg2011direct`:

$$
\lambda(t_f) = -\sum_k \mu_k D_{k,N}
$$

where the sum runs over the collocation nodes of the last segment and $D$ is its differentiation matrix.

**Birkhoff.** The state rates $V$ are design variables, and the state rate defects are $V_j - f_j (dt/d\tau)_j$ with multipliers $\psi_j$.
With $w$ the last row of the Birkhoff matrix of the segment, which integrates the state rates {cite}`costates-ross2024birkhoff`, the estimate is

$$
\lambda_j = -\frac{\psi_j}{w_j}
$$

On a Chebyshev-Gauss-Lobatto grid these weights differ from the node weights stored in the grid data, which belong to the Chebyshev weight function.
The costates at the two ends of the phase come from the multipliers $\omega$ of the state defects:
$\lambda(t_f) = \omega_b$, the multiplier of the defect that defines the final state, and $\lambda(t_0) = \omega_b - \sum_i \omega_i$.

No factor for the duration of the phase or for `defect_ref` is needed, because $dt/d\tau$ is part of the defects and the multipliers are taken in physical units.

## Usage

```{eval-rst}
    .. autofunction:: dymos.utils.costates.estimate_costates
        :noindex:
```

The estimates are returned at the nodes where they are computed: the collocation nodes for Radau and Gauss-Lobatto, and all nodes for Birkhoff.
They are not interpolated to other nodes.
States whose defects are not constraints of the driver, such as states with `solve_segments`, are left out.

The estimates are only as accurate as the optimality conditions at the point where the optimizer stops.
Use a tight optimizer tolerance, as in the example below.
SLSQP stops on a small change of the objective, which can happen before the optimality conditions hold closely; running the driver once more from the solution helps in that case.

## Example: a problem with a known costate

This is example 1 of Garg et al. {cite}`costates-garg2011direct`:

$$
\begin{aligned}
\min \quad & J = \frac{1}{2} \int_0^5 \left( y + u^2 \right) dt \\
\text{s.t.} \quad & \dot{y} = 2 y + 2 u \sqrt{y}, \quad y(0) = 2, \quad y(5) = 1
\end{aligned}
$$

With $x = \sqrt{y}$ the problem becomes linear-quadratic, and its solution is
$x(t) = a e^{\sqrt{2} t} + b e^{-\sqrt{2} t}$ with the costate $p = x - \dot{x}$ of $x$.
The costate of $y$ is $\lambda_y = p / (2 x)$.
The integrated cost is carried as the state `J`, so its costate is one.

The problem is solved below with the Radau and the Birkhoff transcriptions, and the estimates are compared with the exact costate.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import openmdao.api as om
import dymos as dm
from dymos.utils.costates import estimate_costates


class GargODE(om.ExplicitComponent):

    def initialize(self):
        self.options.declare('num_nodes', types=int)

    def setup(self):
        nn = self.options['num_nodes']
        ar = np.arange(nn)
        self.add_input('y', shape=(nn,))
        self.add_input('u', shape=(nn,))
        self.add_output('y_dot', shape=(nn,), units='1/s')
        self.add_output('L', shape=(nn,), units='1/s')
        self.declare_partials('y_dot', ['y', 'u'], rows=ar, cols=ar)
        self.declare_partials('L', 'y', rows=ar, cols=ar, val=0.5)
        self.declare_partials('L', 'u', rows=ar, cols=ar)

    def compute(self, inputs, outputs):
        y, u = inputs['y'], inputs['u']
        outputs['y_dot'] = 2 * y + 2 * u * np.sqrt(y)
        outputs['L'] = 0.5 * (y + u ** 2)

    def compute_partials(self, inputs, partials):
        y, u = inputs['y'], inputs['u']
        partials['y_dot', 'y'] = 2 + u / np.sqrt(y)
        partials['y_dot', 'u'] = 2 * np.sqrt(y)
        partials['L', 'u'] = u


def exact_costate(t, y0=2.0, yf=1.0, tf=5.0):
    r = np.sqrt(2.0)
    den = np.exp(r * tf) - np.exp(-r * tf)
    a = (np.sqrt(yf) - np.sqrt(y0) * np.exp(-r * tf)) / den
    b = (np.sqrt(y0) * np.exp(r * tf) - np.sqrt(yf)) / den
    x = a * np.exp(r * t) + b * np.exp(-r * t)
    p = a * (1 - r) * np.exp(r * t) + b * (1 + r) * np.exp(-r * t)
    return p / (2 * x)


def solve(tx):
    p = om.Problem()
    p.driver = om.ScipyOptimizeDriver(optimizer='SLSQP', tol=1.0E-12, maxiter=500)
    p.driver.options['disp'] = False
    p.driver.declare_coloring()

    traj = p.model.add_subsystem('traj', dm.Trajectory())
    phase = traj.add_phase('phase0', dm.Phase(ode_class=GargODE, transcription=tx))
    phase.set_time_options(fix_initial=True, fix_duration=True)
    phase.add_state('y', fix_initial=True, rate_source='y_dot', targets=['y'], lower=1.0E-3)
    phase.add_state('J', fix_initial=True, rate_source='L')
    phase.add_control('u', targets=['u'])
    phase.add_boundary_constraint('y', loc='final', equals=1.0)
    phase.add_objective('J', loc='final')

    p.setup()

    phase.set_time_val(initial=0.0, duration=5.0)
    phase.set_state_val('y', [2.0, 1.0])
    phase.set_state_val('J', [0.0, 1.0])
    phase.set_control_val('u', [-1.0, 1.0])

    dm.run_problem(p, simulate=False, make_plots=False)
    return p


results = {}
for name, tx in [('Radau, 10 segments of order 3', dm.Radau(num_segments=10, order=3)),
                 ('Birkhoff, 20 LGL nodes', dm.Birkhoff(num_nodes=20, grid_type='lgl'))]:
    p = solve(tx)
    results[name] = estimate_costates(p)['traj.phases.phase0']

t = np.linspace(0, 5, 200)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, exact_costate(t), 'k-', label='exact')
for (name, res), marker in zip(results.items(), ['o', 'x']):
    ax.plot(res['time'], res['costates']['y'], marker, label=name)
ax.set_xlabel('time (s)')
ax.set_ylabel(r'$\lambda_y$')
ax.legend()
plt.show()

for name, res in results.items():
    error = np.abs(res['costates']['y'].ravel() - exact_costate(res['time'])).max()
    print(f"{name}: largest error of lambda_y {error:.1e}, "
          f"largest deviation of lambda_J from one {np.abs(res['costates']['J'] - 1.0).max():.1e}")

In [ ]:
from openmdao.utils.assert_utils import assert_near_equal

for res in results.values():
    assert_near_equal(res['costates']['y'].ravel(), exact_costate(res['time']), tolerance=2.0E-2)
    assert_near_equal(res['costates']['J'].ravel(), np.ones(len(res['time'])), tolerance=1.0E-6)

## References

```{bibliography}
:filter: docname in docnames
:labelprefix: CE
:keyprefix: costates-
```